# Dublin Bikes Availability Estimation Notebook

This notebook uses **`num_bikes_available` as the dependent variable** and treats the task as **bike availability estimation at a target time point**.

Two feature setups are evaluated:
- **Short-state estimation**: uses recent bike lags (`lag_1`, `lag_3`, `lag_6`) plus time/weather/station features.
- **Long-state estimation**: uses `lag_24h` plus time/weather/station features.

The train/test split is **time-based** rather than random.

The model comparison now includes four methods:
1. **Ridge Regression**
2. **Decision Tree**
3. **Random Forest**
4. **XGBoost**


In [2]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
!pip install xgboost
from xgboost import XGBRegressor


   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
    --------------------------------------- 2.4/101.7 MB 13.4 MB/s eta 0:00:08
   -- ------------------------------------- 5.5/101.7 MB 15.2 MB/s eta 0:00:07
   --- ------------------------------------ 9.2/101.7 MB 16.3 MB/s eta 0:00:06
   ----- ---------------------------------- 12.8/101.7 MB 17.1 MB/s eta 0:00:06
   ------ --------------------------------- 16.5/101.7 MB 16.8 MB/s eta 0:00:06
   ------- -------------------------------- 19.9/101.7 MB 16.8 MB/s eta 0:00:05
   --------- ------------------------------ 23.1/101.7 MB 16.8 MB/s eta 0:00:05
   ---------- ----------------------------- 26.5/101.7 MB 16.6 MB/s eta 0:00:05
   ----------- ---------------------------- 30.4/101.7 MB 16.8 MB/s eta 0:00:05
   ------------- -------------------------- 34.6/101.7 MB 17.0 MB/s eta 0:00:04
   --------------- ------------------------ 38.3/101.7 MB 17.1 MB/s eta 0:00:04
   ---------------- ----------------------- 41.9/101

## 1. Load data

In [4]:
# Update the file path if needed
df = pd.read_csv('final_merged_data.csv')
df.head()

,last_reported,station_id,num_bikes_available,num_docks_available,is_installed,is_renting,is_returning,name,address,lat,...,min_humidity_quality_indicator,min_relative_humidity_percent,humidity_std_quality_indicator,relative_humidity_std_deviation,max_pressure_quality_indicator,max_barometric_pressure_hpa,min_pressure_quality_indicator,min_barometric_pressure_hpa,pressure_std_quality_indicator,barometric_pressure_std_deviation
0,2024-12-01 00:10:00,10,15,1,True,True,True,DAME STREET,Dame Street,53.344006,...,0,83.2,0,0.284,0,1002.56,0,1002.26,0,0.083
1,2024-12-01 00:10:00,100,17,8,True,True,True,HEUSTON BRIDGE (SOUTH),Heuston Bridge (South),53.347107,...,0,83.2,0,0.284,0,1002.56,0,1002.26,0,0.083
2,2024-12-01 00:10:00,109,20,9,True,True,True,BUCKINGHAM STREET LOWER,Buckingham Street Lower,53.353333,...,0,83.2,0,0.284,0,1002.56,0,1002.26,0,0.083
3,2024-12-01 00:10:00,11,1,29,True,True,True,EARLSFORT TERRACE,Earlsfort Terrace,53.334293,...,0,83.2,0,0.284,0,1002.56,0,1002.26,0,0.083
4,2024-12-01 00:10:00,114,4,36,True,True,True,WILTON TERRACE (PARK),Wilton Terrace (Park),53.333652,...,0,83.2,0,0.284,0,1002.56,0,1002.26,0,0.083


## 2. Basic preprocessing

In [6]:
df['last_reported'] = pd.to_datetime(df['last_reported'], errors='coerce')
df = df.dropna(subset=['last_reported']).copy()

df = df.sort_values(['station_id', 'last_reported']).reset_index(drop=True)

# Basic time features
df['hour'] = df['last_reported'].dt.hour
df['day_of_week'] = df['last_reported'].dt.dayofweek
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

df[['last_reported', 'station_id', 'num_bikes_available', 'hour', 'day_of_week']].head()

,last_reported,station_id,num_bikes_available,hour,day_of_week
0,2024-12-01 00:20:00,1,21,0,6
1,2024-12-01 00:30:00,1,20,0,6
2,2024-12-01 05:00:00,1,20,5,6
3,2024-12-01 06:10:00,1,20,6,6
4,2024-12-01 06:40:00,1,22,6,6


In [7]:
# Datetime and time features
df["last_reported"] = pd.to_datetime(df["last_reported"])
df = df.sort_values(["station_id", "last_reported"]).copy()

df["hour"] = df["last_reported"].dt.hour
df["day_of_week"] = df["last_reported"].dt.dayofweek

df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

# Weather aggregates
df["temp_avg"] = (df["max_air_temperature_celsius"] + df["min_air_temperature_celsius"]) / 2
df["temp_sq"] = df["temp_avg"] ** 2
df["humidity_avg"] = (df["max_relative_humidity_percent"] + df["min_relative_humidity_percent"]) / 2
df["humidity_sq"] = df["humidity_avg"] ** 2
df["pressure_avg"] = (df["max_barometric_pressure_hpa"] + df["min_barometric_pressure_hpa"]) / 2

## 3. Create lag features

In [9]:
# 10-minute frequency lags
df['lag_1'] = df.groupby('station_id')['num_bikes_available'].shift(1)      # 10 min
df['lag_3'] = df.groupby('station_id')['num_bikes_available'].shift(3)      # 30 min
df['lag_6'] = df.groupby('station_id')['num_bikes_available'].shift(6)      # 60 min
df['lag_24h'] = df.groupby('station_id')['num_bikes_available'].shift(144)  # 24 hours

df[['station_id', 'last_reported', 'num_bikes_available', 'lag_1', 'lag_3', 'lag_6', 'lag_24h']].head(10)

,station_id,last_reported,num_bikes_available,lag_1,lag_3,lag_6,lag_24h
0,1,2024-12-01 00:20:00,21,NaN,NaN,NaN,NaN
1,1,2024-12-01 00:30:00,20,21.0,NaN,NaN,NaN
2,1,2024-12-01 05:00:00,20,20.0,NaN,NaN,NaN
3,1,2024-12-01 06:10:00,20,20.0,21.0,NaN,NaN
4,1,2024-12-01 06:40:00,22,20.0,20.0,NaN,NaN
5,1,2024-12-01 06:50:00,23,22.0,20.0,NaN,NaN
6,1,2024-12-01 07:20:00,24,23.0,20.0,21.0,NaN
7,1,2024-12-01 07:40:00,25,24.0,22.0,20.0,NaN
8,1,2024-12-01 07:50:00,27,25.0,23.0,20.0,NaN
9,1,2024-12-01 08:40:00,30,27.0,24.0,20.0,NaN


## 4. Check available columns

In [11]:
print(df.columns.tolist())

['last_reported', 'station_id', 'num_bikes_available', 'num_docks_available', 'is_installed', 'is_renting', 'is_returning', 'name', 'address', 'lat', 'lon', 'capacity', 'stno', 'year', 'month', 'day', 'hour', 'minute', 'max_air_temp_quality_indicator', 'max_air_temperature_celsius', 'min_air_temp_quality_indicator', 'min_air_temperature_celsius', 'air_temp_std_quality_indicator', 'air_temperature_std_deviation', 'max_grass_temp_quality_indicator', 'max_grass_temperature_celsius', 'min_grass_temp_quality_indicator', 'min_grass_temperature_celsius', 'grass_temp_std_quality_indicator', 'grass_temperature_std_deviation', 'max_soil_temp_5cm_quality_indicator', 'max_soil_temperature_5cm_celsius', 'min_soil_temp_5cm_quality_indicator', 'min_soil_temperature_5cm_celsius', 'soil_temp_std_5cm_quality_indicator', 'soil_temperature_std_deviation_5cm', 'max_soil_temp_10cm_quality_indicator', 'max_soil_temperature_10cm_celsius', 'min_soil_temp_10cm_quality_indicator', 'min_soil_temperature_10cm_cels

## 5. Define helper functions

In [13]:
def evaluate(y_true, y_pred):
    return {
        'R2': r2_score(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAE': mean_absolute_error(y_true, y_pred)
    }

def time_split(df_model, time_col='last_reported', cutoff='2024-12-25'):
    df_model = df_model.sort_values(time_col).copy()
    train = df_model[df_model[time_col] < cutoff].copy()
    test = df_model[df_model[time_col] >= cutoff].copy()
    return train, test

def build_xy(train, test, feature_cols, target='num_bikes_available'):
    X_train = train[feature_cols].copy()
    X_test = test[feature_cols].copy()
    y_train = train[target].copy()
    y_test = test[target].copy()
    return X_train, X_test, y_train, y_test

def run_ridge(train, test, feature_cols, target='num_bikes_available'):
    X_train, X_test, y_train, y_test = build_xy(train, test, feature_cols, target)

    categorical_cols = ['station_id', 'day_of_week']
    numeric_cols = [col for col in feature_cols if col not in categorical_cols]

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numeric_cols),
            ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols)
        ]
    )

    ridge = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', RidgeCV(alphas=np.logspace(-3, 3, 13)))
    ])

    ridge.fit(X_train, y_train)
    pred = ridge.predict(X_test)
    metrics = evaluate(y_test, pred)
    return ridge, pred, metrics

def run_tree_rf_xgb(train, test, feature_cols, target='num_bikes_available', random_state=42):
    X_train = pd.get_dummies(train[feature_cols], columns=['station_id', 'day_of_week'], drop_first=True)
    X_test = pd.get_dummies(test[feature_cols], columns=['station_id', 'day_of_week'], drop_first=True)
    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

    y_train = train[target]
    y_test = test[target]

    tree = DecisionTreeRegressor(max_depth=10, random_state=random_state)
    tree.fit(X_train, y_train)
    tree_pred = tree.predict(X_test)
    tree_metrics = evaluate(y_test, tree_pred)

    rf = RandomForestRegressor(
        n_estimators=300,
        max_depth=12,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=random_state,
        n_jobs=-1
    )
    rf.fit(X_train, y_train)
    rf_pred = rf.predict(X_test)
    rf_metrics = evaluate(y_test, rf_pred)

    xgb = XGBRegressor(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.0,
        reg_lambda=1.0,
        objective='reg:squarederror',
        random_state=random_state,
        n_jobs=-1
    )
    xgb.fit(X_train, y_train)
    xgb_pred = xgb.predict(X_test)
    xgb_metrics = evaluate(y_test, xgb_pred)

    return tree, tree_metrics, rf, rf_metrics, xgb, xgb_metrics

def get_ridge_coefficients(ridge_pipeline, feature_cols):
    preprocessor = ridge_pipeline.named_steps['preprocessor']
    model = ridge_pipeline.named_steps['model']

    categorical_cols = ['station_id', 'day_of_week']
    numeric_cols = [col for col in feature_cols if col not in categorical_cols]

    cat_features = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols).tolist()
    all_features = numeric_cols + cat_features

    coef_df = pd.DataFrame({
        'feature': all_features,
        'coefficient': model.coef_
    })
    coef_df['abs_coefficient'] = coef_df['coefficient'].abs()
    return coef_df.sort_values('abs_coefficient', ascending=False)

def get_model_matrix(train, test, feature_cols):
    X_train = pd.get_dummies(train[feature_cols], columns=['station_id', 'day_of_week'], drop_first=True)
    X_test = pd.get_dummies(test[feature_cols], columns=['station_id', 'day_of_week'], drop_first=True)
    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
    return X_train, X_test


## 6. Short-state 10-min estimation

Short-state estimation keeps time/weather/station features and uses recent bike lags (`lag_1`, `lag_3`, `lag_6`).

In [15]:
features_lag1 = [
    'hour_sin', 'hour_cos',
    'day_of_week',
    'temp_avg', 'temp_sq', 'humidity_avg', 'humidity_sq', 'pressure_avg',
    'lag_1',
    'station_id'
]

df_lag1 = df[features_lag1 + ['num_bikes_available', 'last_reported']].dropna().copy()
train_lag1, test_lag1 = time_split(df_lag1, cutoff='2024-12-25')

ridge_model_lag1, ridge_pred_lag1, ridge_metrics_lag1 = run_ridge(
    train_lag1, test_lag1, features_lag1
)

tree_model_lag1, tree_metrics_lag1, rf_model_lag1, rf_metrics_lag1, xgb_model_lag1, xgb_metrics_lag1 = run_tree_rf_xgb(
    train_lag1, test_lag1, features_lag1
)

pd.DataFrame([
    {'model': 'Ridge', 'setup': 'lag_1', **ridge_metrics_lag1},
    {'model': 'DecisionTree', 'setup': 'lag_1', **tree_metrics_lag1},
    {'model': 'RandomForest', 'setup': 'lag_1', **rf_metrics_lag1},
    {'model': 'XGBoost', 'setup': 'lag_1', **xgb_metrics_lag1},
]).sort_values('R2', ascending=False).reset_index(drop=True)


,model,setup,R2,RMSE,MAE
0,Ridge,lag_1,0.989198,0.942039,0.395422
1,RandomForest,lag_1,0.988572,0.968915,0.402430
2,XGBoost,lag_1,0.987374,1.018440,0.468554
3,DecisionTree,lag_1,0.986408,1.056677,0.414960


In [16]:
print("Best alpha:", ridge_model_lag1.named_steps['model'].alpha_)

ridge_coef_lag1 = get_ridge_coefficients(ridge_model_lag1, features_lag1)
display(ridge_coef_lag1.head(15))


Best alpha: 3.1622776601683795


,feature,coefficient,abs_coefficient
7,lag_1,9.789640,9.789640
56,station_id_51,-0.314304,0.314304
75,station_id_70,-0.297793,0.297793
18,station_id_12,-0.268292,0.268292
64,station_id_59,-0.266415,0.266415
36,station_id_30,-0.264666,0.264666
84,station_id_79,-0.261963,0.261963
8,station_id_2,-0.258672,0.258672
21,station_id_15,-0.255190,0.255190
109,station_id_105,-0.254258,0.254258


## 7. Short-state 30-min estimation

In [18]:
features_lag3 = [
    'hour_sin', 'hour_cos',
    'day_of_week',
    'temp_avg', 'temp_sq', 'humidity_avg', 'humidity_sq', 'pressure_avg',
    'lag_3',
    'station_id'
]

df_lag3 = df[features_lag3 + ['num_bikes_available', 'last_reported']].dropna().copy()
train_lag3, test_lag3 = time_split(df_lag3, cutoff='2024-12-25')

ridge_model_lag3, ridge_pred_lag3, ridge_metrics_lag3 = run_ridge(
    train_lag3, test_lag3, features_lag3
)

tree_model_lag3, tree_metrics_lag3, rf_model_lag3, rf_metrics_lag3, xgb_model_lag3, xgb_metrics_lag3 = run_tree_rf_xgb(
    train_lag3, test_lag3, features_lag3
)

pd.DataFrame([
    {'model': 'Ridge', 'setup': 'lag_3', **ridge_metrics_lag3},
    {'model': 'DecisionTree', 'setup': 'lag_3', **tree_metrics_lag3},
    {'model': 'RandomForest', 'setup': 'lag_3', **rf_metrics_lag3},
    {'model': 'XGBoost', 'setup': 'lag_3', **xgb_metrics_lag3},
]).sort_values('R2', ascending=False).reset_index(drop=True)


,model,setup,R2,RMSE,MAE
0,Ridge,lag_3,0.964114,1.716992,0.924010
1,RandomForest,lag_3,0.961633,1.775365,0.908233
2,XGBoost,lag_3,0.960838,1.793673,0.984355
3,DecisionTree,lag_3,0.957300,1.872928,0.915846


In [19]:
print("Best alpha:", ridge_model_lag3.named_steps['model'].alpha_)

ridge_coef_lag3 = get_ridge_coefficients(ridge_model_lag3, features_lag3)
display(ridge_coef_lag3.head(15))


Best alpha: 0.31622776601683794


,feature,coefficient,abs_coefficient
7,lag_3,9.390919,9.390919
75,station_id_70,-1.216758,1.216758
56,station_id_51,-1.213582,1.213582
64,station_id_59,-1.060083,1.060083
36,station_id_30,-1.050259,1.050259
18,station_id_12,-1.049813,1.049813
84,station_id_79,-1.040522,1.040522
21,station_id_15,-1.018789,1.018789
109,station_id_105,-1.017774,1.017774
8,station_id_2,-1.016200,1.016200


## 8. Short-state 60-min estimation

In [21]:
features_lag6 = [
    'hour_sin', 'hour_cos',
    'day_of_week',
    'temp_avg', 'temp_sq', 'humidity_avg', 'humidity_sq', 'pressure_avg',
    'lag_6',
    'station_id'
]

df_lag6 = df[features_lag6 + ['num_bikes_available', 'last_reported']].dropna().copy()
train_lag6, test_lag6 = time_split(df_lag6, cutoff='2024-12-25')

ridge_model_lag6, ridge_pred_lag6, ridge_metrics_lag6 = run_ridge(
    train_lag6, test_lag6, features_lag6
)

tree_model_lag6, tree_metrics_lag6, rf_model_lag6, rf_metrics_lag6, xgb_model_lag6, xgb_metrics_lag6 = run_tree_rf_xgb(
    train_lag6, test_lag6, features_lag6
)

pd.DataFrame([
    {'model': 'Ridge', 'setup': 'lag_6', **ridge_metrics_lag6},
    {'model': 'DecisionTree', 'setup': 'lag_6', **tree_metrics_lag6},
    {'model': 'RandomForest', 'setup': 'lag_6', **rf_metrics_lag6},
    {'model': 'XGBoost', 'setup': 'lag_6', **xgb_metrics_lag6},
]).sort_values('R2', ascending=False).reset_index(drop=True)


,model,setup,R2,RMSE,MAE
0,Ridge,lag_6,0.926192,2.462401,1.526496
1,XGBoost,lag_6,0.919462,2.572220,1.583264
2,RandomForest,lag_6,0.916585,2.617765,1.517311
3,DecisionTree,lag_6,0.898979,2.880805,1.570538


In [22]:
print("Best alpha:", ridge_model_lag6.named_steps['model'].alpha_)

ridge_coef_lag6 = get_ridge_coefficients(ridge_model_lag6, features_lag6)
display(ridge_coef_lag6.head(15))


Best alpha: 0.1


,feature,coefficient,abs_coefficient
7,lag_6,8.710888,8.710888
75,station_id_70,-2.757973,2.757973
56,station_id_51,-2.725189,2.725189
64,station_id_59,-2.410217,2.410217
36,station_id_30,-2.385155,2.385155
84,station_id_79,-2.364744,2.364744
18,station_id_12,-2.364512,2.364512
109,station_id_105,-2.320731,2.320731
21,station_id_15,-2.318491,2.318491
8,station_id_2,-2.294625,2.294625


## 9. Long-state estimation setup

Long-state estimation keeps time/weather/station features and uses only `lag_24h` as the lagged bike state variable.

In [24]:
features_long = [
    'hour_sin', 'hour_cos',
    'day_of_week',
    'temp_avg', 'temp_sq', 'humidity_avg', 'humidity_sq', 'pressure_avg',
    'lag_24h',
    'station_id'
]

df_long = df[features_long + ['num_bikes_available', 'last_reported']].dropna().copy()
train_long, test_long = time_split(df_long, cutoff='2024-12-25')

print(train_long.shape, test_long.shape)


(219103, 12) (63368, 12)


## 10. Long-state estimation

In [ ]:
ridge_model_long, ridge_pred_long, ridge_metrics_long = run_ridge(
    train_long, test_long, features_long
)
tree_model_long, tree_metrics_long, rf_model_long, rf_metrics_long, xgb_model_long, xgb_metrics_long = run_tree_rf_xgb(
    train_long, test_long, features_long
)

pd.DataFrame([
    {'model': 'Ridge', 'setup': 'lag_24h', **ridge_metrics_long},
    {'model': 'DecisionTree', 'setup': 'lag_24h', **tree_metrics_long},
    {'model': 'RandomForest', 'setup': 'lag_24h', **rf_metrics_long},
    {'model': 'XGBoost', 'setup': 'lag_24h', **xgb_metrics_long},
]).sort_values('R2', ascending=False).reset_index(drop=True)


In [ ]:
print("Best alpha:", ridge_model_long.named_steps['model'].alpha_)

ridge_coef_long = get_ridge_coefficients(ridge_model_long, features_long)
display(ridge_coef_long.head(15))


## 11. Combined comparison across all four methods

In [ ]:
all_results = pd.DataFrame([
    {'model': 'Ridge', 'setup': 'lag_1', **ridge_metrics_lag1},
    {'model': 'DecisionTree', 'setup': 'lag_1', **tree_metrics_lag1},
    {'model': 'RandomForest', 'setup': 'lag_1', **rf_metrics_lag1},
    {'model': 'XGBoost', 'setup': 'lag_1', **xgb_metrics_lag1},

    {'model': 'Ridge', 'setup': 'lag_3', **ridge_metrics_lag3},
    {'model': 'DecisionTree', 'setup': 'lag_3', **tree_metrics_lag3},
    {'model': 'RandomForest', 'setup': 'lag_3', **rf_metrics_lag3},
    {'model': 'XGBoost', 'setup': 'lag_3', **xgb_metrics_lag3},

    {'model': 'Ridge', 'setup': 'lag_6', **ridge_metrics_lag6},
    {'model': 'DecisionTree', 'setup': 'lag_6', **tree_metrics_lag6},
    {'model': 'RandomForest', 'setup': 'lag_6', **rf_metrics_lag6},
    {'model': 'XGBoost', 'setup': 'lag_6', **xgb_metrics_lag6},

    {'model': 'Ridge', 'setup': 'lag_24h', **ridge_metrics_long},
    {'model': 'DecisionTree', 'setup': 'lag_24h', **tree_metrics_long},
    {'model': 'RandomForest', 'setup': 'lag_24h', **rf_metrics_long},
    {'model': 'XGBoost', 'setup': 'lag_24h', **xgb_metrics_long},
])

all_results.sort_values(['setup', 'R2'], ascending=[True, False]).reset_index(drop=True)
